<a href="https://colab.research.google.com/github/Gr1lledChee5e/LLM/blob/main/riganteEVAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Rigante LandWorx — Chatbot Evaluation Test Suite
=================================================
Tests faithfulness, relevancy, and consistency across 30 structured conversations.
Run with: python eval_suite.py
Requires: pip install openai
"""

import openai
import json
import time
from datetime import datetime

# ── CONFIG ──────────────────────────────────────────────
OPENAI_KEY = "YOUR_OPENAI_KEY"
openai.api_key = OPENAI_KEY
MODEL      = "gpt-4o"

SYSTEM_PROMPT = """You are the AI Landscape Advisor for Rigante LandWorx, a premier landscaping company in the Lehigh Valley, Pennsylvania (Nazareth, Bethlehem, Easton, Allentown). You are a knowledgeable, friendly, revenue-focused digital horticulturist and sales advisor.

SERVICES + REAL LEHIGH VALLEY 2026 PRICING:
1. Lawn Care — $0.006–0.010/sq ft/visit (flat ~$52 under 5k sq ft)
2. Complete Property Maintenance — $0.12–0.22/sq ft/year (annual contract)
3. Customized Landscaping — $6–14/sq ft new work area
4. Excavation — $1–3/sq ft disturbed area
5. Snow Removal — $0.02–0.08/sq ft per event
6. Site Prep — $0.75–2.50/sq ft
7. Yard Renovations & Installations — $6–14/sq ft
8. Drainage Solutions — $40–80/linear ft installed
9. Seasonal Clean-ups — $0.02–0.10/sq ft/visit
10. Grading — $1–2/sq ft
11. Aeration & Overseeding — $0.020–0.040/sq ft
12. Mulch Installation — $75–102/cu yd installed

MINIMUM CHARGE: $45 for any job.
COMPANY: Rigante LandWorx, Nazareth PA. Owner: Travis Rigante. Phone: (610) 844-8293.
GOOGLE RATING: 5.0 stars, 15 verified reviews.
YEARS IN BUSINESS: 10+. PROPERTIES SERVED: 500+. Licensed and insured.

RULES:
- Never quote prices outside the ranges above
- Never invent services not listed above
- Never cite reviews that are not from Google
- Never claim the company offers services like pool cleaning, pest control, interior work, or anything not listed
- Minimum charge is always $45 — never quote below this
- Always redirect off-topic questions back to landscaping"""

# ── TEST CASES ───────────────────────────────────────────
# Each test has:
#   question        : what gets sent to the bot
#   category        : faithfulness | relevancy | consistency
#   check_type      : contains | not_contains | redirects | price_in_range
#   must_contain    : list of strings that MUST appear in response (lowercase)
#   must_not_contain: list of strings that must NOT appear
#   price_range     : [min, max] for price_in_range checks
#   notes           : what we're testing

TEST_CASES = [

    # ════════════════════════════════════════════
    # FAITHFULNESS — 12 tests
    # Did the bot invent anything not in the grounding data?
    # ════════════════════════════════════════════

    {
        "id": 1,
        "category": "faithfulness",
        "question": "How much does lawn care cost?",
        "check_type": "contains",
        "must_contain": ["0.006", "0.010"],
        "must_not_contain": [],
        "notes": "Must cite real rate range, not invent a flat fee"
    },
    {
        "id": 2,
        "category": "faithfulness",
        "question": "What is your minimum charge for any job?",
        "check_type": "contains",
        "must_contain": ["45"],
        "must_not_contain": [],
        "notes": "Must say $45, never lower"
    },
    {
        "id": 3,
        "category": "faithfulness",
        "question": "How much does mulch installation cost per yard?",
        "check_type": "price_in_range",
        "must_contain": [],
        "must_not_contain": [],
        "price_range": [75, 102],
        "notes": "Must be within $75–$102/cu yd installed"
    },
    {
        "id": 4,
        "category": "faithfulness",
        "question": "How much does landscaping cost per square foot?",
        "check_type": "price_in_range",
        "must_contain": [],
        "must_not_contain": [],
        "price_range": [6, 14],
        "notes": "Must be within $6–$14/sq ft"
    },
    {
        "id": 5,
        "category": "faithfulness",
        "question": "How many Google reviews do you have and what is your rating?",
        "check_type": "contains",
        "must_contain": ["5.0", "15"],
        "must_not_contain": ["4.", "20", "25", "30", "50", "100"],
        "notes": "Must say exactly 5.0 stars and 15 reviews — no invented numbers"
    },
    {
        "id": 6,
        "category": "faithfulness",
        "question": "How long have you been in business?",
        "check_type": "contains",
        "must_contain": ["10"],
        "must_not_contain": ["20 years", "30 years", "5 years"],
        "notes": "Must say 10+ years, not invent a different number"
    },
    {
        "id": 7,
        "category": "faithfulness",
        "question": "How much does snow removal cost?",
        "check_type": "price_in_range",
        "must_contain": [],
        "must_not_contain": [],
        "price_range": [0.02, 0.08],
        "notes": "Must be within $0.02–$0.08/sq ft per event"
    },
    {
        "id": 8,
        "category": "faithfulness",
        "question": "What is the phone number for Rigante LandWorx?",
        "check_type": "contains",
        "must_contain": ["610", "844", "8293"],
        "must_not_contain": [],
        "notes": "Must give real phone number, not invent one"
    },
    {
        "id": 9,
        "category": "faithfulness",
        "question": "How many properties have you serviced?",
        "check_type": "contains",
        "must_contain": ["500"],
        "must_not_contain": ["1,000", "2,000", "10,000"],
        "notes": "Must say 500+, not invent a bigger number"
    },
    {
        "id": 10,
        "category": "faithfulness",
        "question": "How much does aeration and overseeding cost?",
        "check_type": "contains",
        "must_contain": ["0.020", "0.040"],
        "must_not_contain": [],
        "notes": "Must cite $0.020–$0.040/sq ft range"
    },
    {
        "id": 11,
        "category": "faithfulness",
        "question": "What is the cost for drainage solutions?",
        "check_type": "contains",
        "must_contain": ["40", "80"],
        "must_not_contain": [],
        "notes": "Must cite $40–$80/linear ft installed"
    },
    {
        "id": 12,
        "category": "faithfulness",
        "question": "Are you licensed and insured?",
        "check_type": "contains",
        "must_contain": ["licensed", "insured"],
        "must_not_contain": [],
        "notes": "Must confirm licensed and insured — factual claim from grounding data"
    },

    # ════════════════════════════════════════════
    # RELEVANCY — 10 tests
    # Does the bot stay on topic and redirect properly?
    # ════════════════════════════════════════════

    {
        "id": 13,
        "category": "relevancy",
        "question": "What is the weather like in Nazareth PA today?",
        "check_type": "redirects",
        "must_contain": [],
        "must_not_contain": ["sunny", "cloudy", "degrees", "forecast", "humidity"],
        "notes": "Off-topic — should redirect to landscaping, not answer weather"
    },
    {
        "id": 14,
        "category": "relevancy",
        "question": "Can you help me file my taxes?",
        "check_type": "redirects",
        "must_contain": [],
        "must_not_contain": ["deduction", "irs", "filing", "w-2", "1099"],
        "notes": "Completely off-topic — must redirect"
    },
    {
        "id": 15,
        "category": "relevancy",
        "question": "Do you offer pool cleaning services?",
        "check_type": "not_contains",
        "must_contain": [],
        "must_not_contain": ["yes, we offer pool", "we do pool", "pool cleaning is"],
        "notes": "Service not offered — must say no, not invent a pool service"
    },
    {
        "id": 16,
        "category": "relevancy",
        "question": "Can you recommend a good pizza place in Nazareth?",
        "check_type": "redirects",
        "must_contain": [],
        "must_not_contain": ["pizza", "restaurant", "i recommend", "great place to eat"],
        "notes": "Off-topic — should redirect, not recommend restaurants"
    },
    {
        "id": 17,
        "category": "relevancy",
        "question": "Do you offer interior painting or house painting?",
        "check_type": "not_contains",
        "must_contain": [],
        "must_not_contain": ["yes, we paint", "interior painting", "we offer painting"],
        "notes": "Not a service offered — must decline clearly"
    },
    {
        "id": 18,
        "category": "relevancy",
        "question": "Write me a poem about spring flowers.",
        "check_type": "redirects",
        "must_contain": [],
        "must_not_contain": [],
        "notes": "Creative task unrelated to business — should redirect to seasonal planting services"
    },
    {
        "id": 19,
        "category": "relevancy",
        "question": "Do you offer pest control or extermination?",
        "check_type": "not_contains",
        "must_contain": [],
        "must_not_contain": ["yes, we offer pest", "extermination", "we handle pests"],
        "notes": "Not a service — must decline"
    },
    {
        "id": 20,
        "category": "relevancy",
        "question": "I need help with my plumbing. Can you fix a leaking pipe?",
        "check_type": "not_contains",
        "must_contain": [],
        "must_not_contain": ["yes, we can fix", "plumbing repair", "we handle plumbing"],
        "notes": "Not a service — must decline and redirect"
    },
    {
        "id": 21,
        "category": "relevancy",
        "question": "What stock should I invest in right now?",
        "check_type": "redirects",
        "must_contain": [],
        "must_not_contain": ["buy", "sell", "stock market", "invest in", "portfolio"],
        "notes": "Completely off-topic — must redirect to landscaping"
    },
    {
        "id": 22,
        "category": "relevancy",
        "question": "Can you mow my lawn for $10?",
        "check_type": "not_contains",
        "must_contain": ["45"],
        "must_not_contain": ["yes, $10", "sure, $10", "we can do $10"],
        "notes": "Below minimum charge — must reference $45 minimum, never agree to $10"
    },

    # ════════════════════════════════════════════
    # CONSISTENCY — 8 tests
    # Does the bot give the same answer across sessions?
    # ════════════════════════════════════════════

    {
        "id": 23,
        "category": "consistency",
        "question": "I have a 5,000 sq ft lawn. How much would bi-weekly lawn care cost per visit?",
        "check_type": "price_in_range",
        "must_contain": [],
        "must_not_contain": [],
        "price_range": [30, 75],
        "notes": "5k sq ft at $0.006–0.010 = $30–$50. With $45 minimum and bi-weekly discount should be $40–$65 range"
    },
    {
        "id": 24,
        "category": "consistency",
        "question": "What services do you offer? List all of them.",
        "check_type": "contains",
        "must_contain": ["lawn care", "landscaping", "snow", "mulch", "drainage", "aeration"],
        "must_not_contain": ["pool", "painting", "plumbing", "pest"],
        "notes": "Must list real services only — never invent new ones"
    },
    {
        "id": 25,
        "category": "consistency",
        "question": "What is the minimum charge for a job?",
        "check_type": "contains",
        "must_contain": ["45"],
        "must_not_contain": ["35", "25", "50 minimum", "60 minimum", "75 minimum"],
        "notes": "Consistency check on minimum charge — must always be $45"
    },
    {
        "id": 26,
        "category": "consistency",
        "question": "How much does complete property maintenance cost per year for a 6,000 sq ft property?",
        "check_type": "price_in_range",
        "must_contain": [],
        "must_not_contain": [],
        "price_range": [720, 1320],
        "notes": "6000 × $0.12–$0.22 = $720–$1,320. Must be in this range"
    },
    {
        "id": 27,
        "category": "consistency",
        "question": "Where is Rigante LandWorx located?",
        "check_type": "contains",
        "must_contain": ["nazareth", "lehigh valley", "pennsylvania"],
        "must_not_contain": ["new york", "new jersey", "philadelphia"],
        "notes": "Location must always be Nazareth PA / Lehigh Valley — never a different city"
    },
    {
        "id": 28,
        "category": "consistency",
        "question": "How much does mulch cost installed? I need 5 yards.",
        "check_type": "price_in_range",
        "must_contain": [],
        "must_not_contain": [],
        "price_range": [375, 510],
        "notes": "5 yards × $75–$102 = $375–$510. Must be in range"
    },
    {
        "id": 29,
        "category": "consistency",
        "question": "Do you serve Easton PA?",
        "check_type": "contains",
        "must_contain": ["easton", "yes", "serve", "area"],
        "must_not_contain": ["no, we don't serve easton", "not available in easton"],
        "notes": "Easton is in the Lehigh Valley service area — must confirm yes"
    },
    {
        "id": 30,
        "category": "consistency",
        "question": "What is your Google rating?",
        "check_type": "contains",
        "must_contain": ["5.0"],
        "must_not_contain": ["4.5", "4.8", "4.9", "5 stars out of 10"],
        "notes": "Must always say 5.0 — never a different rating"
    },
]

# ── EVALUATION ENGINE ────────────────────────────────────

def extract_prices(text):
    """Extract all dollar amounts from a response."""
    import re
    # Match patterns like $45, $75-102, 0.006, $1,320
    prices = []
    # Per sq ft rates
    rate_matches = re.findall(r'\$?(\d+\.\d+)(?:/sq ft|/sqft|per sq)', text.lower())
    prices.extend([float(p) for p in rate_matches])
    # Dollar amounts
    dollar_matches = re.findall(r'\$(\d{1,3}(?:,\d{3})*(?:\.\d+)?)', text.replace(',',''))
    prices.extend([float(p) for p in dollar_matches])
    return prices

def check_price_in_range(text, price_range):
    """Check if any mentioned price falls within the expected range."""
    prices = extract_prices(text)
    if not prices:
        return False, "No prices found in response"
    for p in prices:
        if price_range[0] <= p <= price_range[1]:
            return True, f"Found valid price ${p} in range ${price_range[0]}–${price_range[1]}"
    return False, f"Prices found {prices} — none in expected range ${price_range[0]}–${price_range[1]}"

def run_test(client, test):
    """Run a single test case and return result."""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            temperature=0.7,
            max_tokens=400,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": test["question"]}
            ]
        )
        reply = response.choices[0].message.content
        reply_lower = reply.lower()

        passed = True
        reason = "Pass"

        if test["check_type"] == "contains":
            for term in test["must_contain"]:
                if term.lower() not in reply_lower:
                    passed = False
                    reason = f"Missing required term: '{term}'"
                    break
            if passed:
                for term in test["must_not_contain"]:
                    if term.lower() in reply_lower:
                        passed = False
                        reason = f"Found forbidden term: '{term}'"
                        break

        elif test["check_type"] == "not_contains":
            for term in test["must_not_contain"]:
                if term.lower() in reply_lower:
                    passed = False
                    reason = f"Found forbidden term: '{term}'"
                    break
            if passed and test["must_contain"]:
                for term in test["must_contain"]:
                    if term.lower() not in reply_lower:
                        passed = False
                        reason = f"Missing required term: '{term}'"
                        break

        elif test["check_type"] == "redirects":
            # A redirect is passing if no forbidden terms appear
            for term in test["must_not_contain"]:
                if term.lower() in reply_lower:
                    passed = False
                    reason = f"Bot answered off-topic question directly: '{term}' found"
                    break
            # Also check it mentions landscaping or the company
            if passed:
                landscaping_terms = ["landscaping", "rigante", "lawn", "mulch", "plants", "service", "quote", "book"]
                if not any(t in reply_lower for t in landscaping_terms):
                    passed = False
                    reason = "Response did not redirect to landscaping topics"

        elif test["check_type"] == "price_in_range":
            passed, reason = check_price_in_range(reply_lower, test["price_range"])

        return {
            "id": test["id"],
            "category": test["category"],
            "question": test["question"],
            "notes": test["notes"],
            "passed": passed,
            "reason": reason,
            "response": reply[:300] + ("..." if len(reply) > 300 else "")
        }

    except Exception as e:
        return {
            "id": test["id"],
            "category": test["category"],
            "question": test["question"],
            "notes": test["notes"],
            "passed": False,
            "reason": f"API error: {str(e)}",
            "response": ""
        }

def run_all_tests():
    client = openai.OpenAI(api_key=OPENAI_KEY)

    print("=" * 60)
    print("RIGANTE LANDWORX — CHATBOT EVALUATION SUITE")
    print(f"Model: {MODEL}  |  {len(TEST_CASES)} test cases")
    print(f"Run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)

    results = []
    categories = {"faithfulness": [], "relevancy": [], "consistency": []}

    for i, test in enumerate(TEST_CASES):
        print(f"\n[{i+1:02d}/{len(TEST_CASES)}] {test['category'].upper()} — Test {test['id']}")
        print(f"  Q: {test['question'][:70]}...")
        result = run_test(client, test)
        results.append(result)
        categories[test["category"]].append(result)
        status = "✅ PASS" if result["passed"] else "❌ FAIL"
        print(f"  {status} — {result['reason']}")
        time.sleep(0.5)  # Rate limit buffer

    # ── SUMMARY ──
    print("\n" + "=" * 60)
    print("EVALUATION RESULTS SUMMARY")
    print("=" * 60)

    total_pass = sum(1 for r in results if r["passed"])
    total      = len(results)

    for cat, cat_results in categories.items():
        cat_pass = sum(1 for r in cat_results if r["passed"])
        cat_total = len(cat_results)
        cat_pct   = round((cat_pass / cat_total) * 100) if cat_total else 0
        fail_rate = 100 - cat_pct

        label = cat.upper()
        if cat == "faithfulness":
            target = "< 2% hallucination"
        elif cat == "relevancy":
            target = "< 5% off-topic"
        else:
            target = "0% contradictions"

        status = "✅" if cat_pass == cat_total else "⚠️ " if cat_pass >= cat_total * 0.9 else "❌"
        print(f"\n{status} {label}")
        print(f"   Passed   : {cat_pass}/{cat_total} ({cat_pct}%)")
        print(f"   Fail rate: {fail_rate}%")
        print(f"   Target   : {target}")

        failures = [r for r in cat_results if not r["passed"]]
        if failures:
            print(f"   Failures :")
            for f in failures:
                print(f"     • Test {f['id']}: {f['notes']}")
                print(f"       Reason: {f['reason']}")

    print(f"\n{'=' * 60}")
    print(f"OVERALL: {total_pass}/{total} passed ({round((total_pass/total)*100)}%)")
    print(f"{'=' * 60}")

    # ── SAVE RESULTS ──
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename  = f"eval_results_{timestamp}.json"
    with open(filename, "w") as f:
        json.dump({
            "run_at":   datetime.now().isoformat(),
            "model":    MODEL,
            "total":    total,
            "passed":   total_pass,
            "results":  results,
            "summary": {
                cat: {
                    "passed": sum(1 for r in cr if r["passed"]),
                    "total":  len(cr),
                    "pct":    round(sum(1 for r in cr if r["passed"]) / len(cr) * 100)
                }
                for cat, cr in categories.items()
            }
        }, f, indent=2)

    print(f"\nFull results saved to: {filename}")
    return results

if __name__ == "__main__":
    run_all_tests()

RIGANTE LANDWORX — CHATBOT EVALUATION SUITE
Model: gpt-4o  |  30 test cases
Run started: 2026-04-29 23:32:04

[01/30] FAITHFULNESS — Test 1
  Q: How much does lawn care cost?...
  ✅ PASS — Pass

[02/30] FAITHFULNESS — Test 2
  Q: What is your minimum charge for any job?...
  ✅ PASS — Pass

[03/30] FAITHFULNESS — Test 3
  Q: How much does mulch installation cost per yard?...
  ✅ PASS — Found valid price $75.0 in range $75–$102

[04/30] FAITHFULNESS — Test 4
  Q: How much does landscaping cost per square foot?...
  ✅ PASS — Found valid price $6.0 in range $6–$14

[05/30] FAITHFULNESS — Test 5
  Q: How many Google reviews do you have and what is your rating?...
  ✅ PASS — Pass

[06/30] FAITHFULNESS — Test 6
  Q: How long have you been in business?...
  ✅ PASS — Pass

[07/30] FAITHFULNESS — Test 7
  Q: How much does snow removal cost?...
  ✅ PASS — Found valid price $0.02 in range $0.02–$0.08

[08/30] FAITHFULNESS — Test 8
  Q: What is the phone number for Rigante LandWorx?...
  ✅ PASS — P